# Notebook 01 — Why agents

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI (Financial Services Industry) Semantic Layer Workshop on AWS — Workshop 2

---

Workshop 1 gave me a knowledge graph and SPARQL queries. Why do I need agents at all?

This notebook answers that question with working code. By the end, you will have
seen the pattern that every Phase 1 agent implements — and you will understand why
that pattern, not a more capable one, is the right choice for a regulated institution.

In [ ]:
# WS2 notebook setup — installs dependencies into THIS kernel.
# uv manages the WS2 virtual environment; this cell installs it
# into the running kernel so imports work without manual setup.
import sys, subprocess, os
from pathlib import Path

# Find the use-case-applications root (3 levels up from phase-1-referral/)
ws2_root = Path(os.getcwd()).parents[1]
venv_python = ws2_root / '.venv' / 'bin' / 'python'

if venv_python.exists() and str(venv_python) != sys.executable:
    # venv exists but we're not running inside it — install into current kernel
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--disable-pip-version-check',
         'rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0',
         'pydantic>=2.0.0'],
        cwd='/tmp'
    )
elif not venv_python.exists():
    # venv not yet created — run uv sync first
    subprocess.check_call(
        ['uv', 'sync', '--all-groups', '--quiet'],
        cwd=str(ws2_root)
    )
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--disable-pip-version-check',
         'rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0',
         'pydantic>=2.0.0'],
        cwd='/tmp'
    )

print('WS2 dependencies ready.')


## Key terms for this notebook

| Term | What it is |
|------|------------|
| **Agent** | A piece of software that receives a request in one form (natural language), translates it into an operation a system can execute (a SPARQL query), runs the operation, and returns the result. In ATLAS, agents are thin translators — they do not reason; they route. |
| **LLM-as-interface** | Using a Large Language Model to translate between the user's language and a system's language — for example, mapping a business question to a SPARQL query template. The LLM handles the surface, not the logic. |
| **LLM-as-reasoner** | Using a Large Language Model to draw conclusions from data — for example, deciding whether a customer is wealth-eligible by reading their transactions. ATLAS never does this. |
| **Template-based SPARQL** | A pre-written, validated SPARQL query that can be selected and parameterised rather than generated fresh each time. Deterministic by construction: the same question always selects the same template, which always produces the same query. |
| **Embedding** | A numeric vector that captures the meaning of a piece of text. Two questions that mean the same thing produce similar vectors; questions that mean different things produce distant vectors. The `nl-to-sparql-agent` uses embeddings to find which ground-truth question is closest in meaning to the user's question, then uses that question's paired SPARQL template. |
| **SR 11-7** | The U.S. Federal Reserve's 2011 supervisory guidance on model risk management. Requires that any model used in a banking decision be documented, independently validated, and explainable. A model that cannot be explained per-record fails this standard. |
| **OCC 2011-12** | The Office of the Comptroller of the Currency's parallel guidance, same year. Together with SR 11-7, these two documents define the baseline compliance posture for every AI model a U.S. bank puts into production. |
| **Determinism** | The property that the same input always produces the same output. SPARQL queries against a stable graph are deterministic. Embedding-based template selection over a fixed template library is deterministic. LLM text generation is not. |

## LLM at the edges, not in the middle

A SPARQL query is already a powerful tool. Given a populated graph, the query
`SELECT ?customer WHERE { ?customer a atlas:Customer ; atlas:memberOf ?hh . FILTER NOT EXISTS { ?customer atlas:hasAdvisor ?rel } }` returns
every customer with no wealth advisor — exactly the kind of question a Consumer
Banker needs answered to identify referral candidates. Workshop 1 proved this
works. The knowledge graph is correct, the queries run, the answers come back.

The problem is not the query. The problem is that a banker does not speak SPARQL.
They ask: *"Which of my customers should I be talking to the wealth team about?"*
That sentence has no `PREFIX`, no `SELECT`, no triple pattern. Bridging the gap
between a business question and a graph query is what an agent does. It receives
the question in the banker's language, finds the right pre-written SPARQL template
from a validated library, executes it against the SLGD (Semantic Layer Graph
Database), and returns the result in a form the banker can read. The agent is a
translator. It is not a reasoner.

The distinction between translator and reasoner is not a design preference — it is
a regulatory requirement. SR 11-7 and OCC 2011-12, the two U.S. federal model risk
management frameworks that govern AI in banking, require that any model used in a
compliance decision be documented, independently validated, and explainable at the
level of an individual record. *"The model said so"* is not an explanation a bank
examiner will accept. *"The SPARQL query `atlas:producesSignal` found a `LargeInboundWire`
signal dated 14 March, evidenced by transaction `txn:82a4f`, with a SHAP
(SHapley Additive exPlanations) score of 0.73 from the XGBoost (Extreme Gradient
Boosting) model registered in the MRM (Model Risk Management) system as
`atlas-xgb-wealth-v2`"* is. The SPARQL query against the SHACL-validated graph
is the audit trail. The LLM is not in that trail.

This is why ATLAS uses LLMs in exactly three places: translating natural language
into SPARQL (`nl-to-sparql-agent`), drafting narrative rationale for a referral
that a human then approves or rejects (`referral-rationale-drafter`), and
summarising market themes for a wealth advisor's reading pane in Phase 2
(`theme-summarizer`). All three are at the edges of the architecture — they handle
surface language in or out, but the reasoning in between is always a deterministic
query against a governed graph. If you removed all three LLM components, the
architecture would still make correct referral decisions. It would just require a
SPARQL-literate human at the keyboard.

Most agentic AI demos go further. They let the model read raw transaction data and
decide which customers are wealth-eligible. This feels more capable — the model can
handle questions that do not match any pre-written template. But it is also
unauditable: given the same transactions on two different days, the model may
produce different assessments, for reasons it cannot explain per-record in the terms
SR 11-7 requires. A bank that built its referral workflow on top of that pattern
would fail a model risk examination. ATLAS's constraint is not a limitation — it is
the architecture that makes the system safe to put in front of a regulator.

In [ ]:
import sys
import os
import json
import math

# Workshop 1's shared helpers — same path as notebook 00.
sys.path.insert(0, "../../../agentic-semantic-layer/notebooks/shared")

import boto3
import yaml
from pathlib import Path

from atlas_neptune import NeptuneClient
from atlas_sparql import build_prefixes

STACK_NAME = "atlas-neptune-twotier"
AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

try:
    REPO_ROOT = Path(__file__).resolve().parents[3]
except NameError:
    REPO_ROOT = Path("../../..").resolve()

GROUND_TRUTH_PATH = REPO_ROOT / "agentic-semantic-layer/prompts/ground-truth.yaml"

# Retrieve Neptune endpoint from CloudFormation — same pattern as notebook 00.
cfn = boto3.client("cloudformation", region_name=AWS_REGION)
response = cfn.describe_stacks(StackName=STACK_NAME)
outputs = {o["OutputKey"]: o["OutputValue"] for o in response["Stacks"][0].get("Outputs", [])}

slgd = NeptuneClient(
    endpoint=outputs["SLGDEndpoint"],
    port=int(outputs.get("SLGDPort", 8182)),
)

# Load the ground-truth NL-to-SPARQL pairs Workshop 1 ships.
# These are the validated templates the nl-to-sparql-agent selects from.
with open(GROUND_TRUTH_PATH) as f:
    ground_truth = yaml.safe_load(f)

pairs = ground_truth["pairs"]

print(f"SLGD endpoint:        {outputs['SLGDEndpoint']}")
print(f"Ground-truth pairs:   {len(pairs)}")
print(f"Ground-truth path:    {GROUND_TRUTH_PATH}")
print()
print("Questions in the ground-truth library:")
for i, pair in enumerate(pairs):
    print(f"  [{i}] {pair['question']}")

In [ ]:
# Cell 1 — A direct SPARQL query against the SLGD.
#
# We pick the Phase 1 pilot question about advisor coverage gaps and run it
# directly. This is Workshop 1's model: you write SPARQL, you run it, you read
# results. It is correct. The next cell shows what changes when a banker who
# does not write SPARQL needs the same answer.

PILOT_QUESTION = "Which customers have no wealth advisor assigned?"

# Find the matching pair in the ground-truth library.
pilot_pair = next(p for p in pairs if p["question"] == PILOT_QUESTION)
pilot_sparql = build_prefixes() + "\n" + pilot_pair["sparql"]

print(f"Question: {PILOT_QUESTION}")
print()
print("SPARQL:")
print(pilot_sparql)
print()

rows = slgd.query(pilot_sparql)

print(f"Results ({len(rows)} rows total, showing first 5):")
if rows:
    headers = list(rows[0].keys())
    print("  " + "  ".join(f"{h:<30}" for h in headers))
    print("  " + "  ".join("-" * 30 for _ in headers))
    for row in rows[:5]:
        print("  " + "  ".join(f"{str(row.get(h, '')):<30}" for h in headers))
else:
    print("  (no rows returned — check that Workshop 1 module 4 ran successfully)")

The query answered the question correctly. But it required us to know SPARQL, to
know the `atlas:hasAdvisor` predicate by name, and to know which of the nine
ground-truth templates matched this business question. The next cell removes all
three requirements — and that removal is what an agent is.

In [ ]:
# Cell 2 — The agent pattern: NL question → embedding lookup → SPARQL execution.
#
# ask_graph() is the pattern every Phase 1 agent implements:
#   1. Embed the incoming question using Amazon Titan Embeddings v2.
#   2. Embed every question in the ground-truth library (cached after the first call).
#   3. Find the ground-truth question whose embedding is closest to the query
#      embedding (cosine similarity).
#   4. Execute that question's paired SPARQL template against the SLGD.
#   5. Return the SPARQL that ran, the results, and the template's ID so the
#      caller can include it in an audit record.
#
# The LLM (Titan Embeddings) does exactly one thing: map text to a vector so we
# can measure semantic similarity. It does not generate SPARQL. It does not decide
# which customers are wealth-eligible. It finds the closest question in a library
# that a human already wrote and validated. That is the LLM-as-interface pattern.

EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v2:0"

bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

def _embed(text: str) -> list:
    """Return the Titan Embeddings v2 vector for a text string."""
    body = json.dumps({"inputText": text})
    response = bedrock_runtime.invoke_model(
        modelId=EMBEDDING_MODEL_ID,
        body=body,
        accept="application/json",
        contentType="application/json",
    )
    return json.loads(response["body"].read())["embedding"]


def _cosine(a: list, b: list) -> float:
    """Cosine similarity between two equal-length vectors."""
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x * x for x in a))
    mag_b = math.sqrt(sum(x * x for x in b))
    return dot / (mag_a * mag_b) if mag_a and mag_b else 0.0


# Embed all ground-truth questions once and cache the vectors.
# In the production nl-to-sparql-agent, this cache is warmed at Lambda cold start.
print(f"Embedding {len(pairs)} ground-truth questions (one Bedrock call each)...")
gt_embeddings = []
for pair in pairs:
    gt_embeddings.append(_embed(pair["question"]))
    print(f"  embedded: {pair['question'][:60]}")
print()
print("Ground-truth embeddings cached.")


def ask_graph(question: str) -> dict:
    """Translate a natural-language question into a SPARQL result.

    Returns a dict with keys:
      sparql       — the SPARQL that was executed
      results      — list of result row dicts
      template_id  — the index of the matched ground-truth pair
      similarity   — cosine similarity of the question to the matched template
    """
    # Step 1: embed the incoming question.
    q_vec = _embed(question)

    # Step 2: find the closest ground-truth question by cosine similarity.
    scores = [_cosine(q_vec, gt_vec) for gt_vec in gt_embeddings]
    best_idx = scores.index(max(scores))
    best_pair = pairs[best_idx]

    # Step 3: execute the matched SPARQL template against the SLGD.
    sparql = build_prefixes() + "\n" + best_pair["sparql"]
    results = slgd.query(sparql)

    return {
        "sparql":      sparql,
        "results":     results,
        "template_id": best_idx,
        "similarity":  round(scores[best_idx], 4),
    }


print("ask_graph() is ready.")

In [ ]:
# Cell 3 — Call ask_graph() with the three Phase 1 pilot questions.
#
# These are the three questions at the bottom of ground-truth.yaml, labelled
# "Phase 1 pilot questions." A Consumer Banker would ask these questions
# naturally; ask_graph() routes each to the correct SPARQL template without
# the caller knowing any SPARQL.

PILOT_QUESTIONS = [
    "Which customers have no wealth advisor assigned?",
    "Which households have mixed wealth coverage?",
    "Who was this customer's advisor 18 months ago?",
]

results_by_question = {}

for question in PILOT_QUESTIONS:
    print("=" * 70)
    print(f"Question: {question}")
    print()

    outcome = ask_graph(question)
    results_by_question[question] = outcome

    print(f"Matched template [{outcome['template_id']}]  "
          f"(similarity: {outcome['similarity']})")
    print()
    print("SPARQL executed:")
    # Print only the query body (after the prefix block) for readability.
    query_body = outcome["sparql"].split("\n\n", 1)[-1]
    for line in query_body.strip().splitlines():
        print(f"  {line}")
    print()

    rows = outcome["results"]
    print(f"Results ({len(rows)} rows total, showing first 3):")
    if rows:
        headers = list(rows[0].keys())
        print("  " + "  ".join(f"{h:<35}" for h in headers))
        print("  " + "  ".join("-" * 35 for _ in headers))
        for row in rows[:3]:
            print("  " + "  ".join(f"{str(row.get(h, '')):<35}" for h in headers))
    else:
        print("  (no rows — check that Workshop 1 module 4 ran successfully)")
    print()

## The determinism test

The most important property of `ask_graph()` is not accuracy — it is determinism.
The same question must produce the same SPARQL on every invocation, because that
is what makes the system auditable. Amazon Titan Embeddings v2 is a deterministic
model: the same input text always produces the same embedding vector. The cosine
similarity ranking over a fixed library of nine templates therefore always selects
the same template. The same template always produces the same SPARQL.

If this test ever fails — if the same question produces different SPARQL across
runs — it would mean `ask_graph()` is calling a generative model somewhere and
producing SPARQL from scratch rather than selecting from a validated library. That
would be the LLM-as-reasoner pattern, which is what ATLAS is designed to avoid.

In [ ]:
# Determinism test — run each pilot question 5 times and assert byte-identical
# SPARQL across all runs.
#
# We compare the full SPARQL string including the prefix block, not just the
# query body. Any variation — even whitespace — is a failure: it indicates
# the template selection is non-deterministic or the SPARQL is being generated
# rather than looked up.

RUNS = 5

print(f"Running each pilot question {RUNS} times and comparing SPARQL output.")
print()

all_deterministic = True

for question in PILOT_QUESTIONS:
    sparql_runs = [ask_graph(question)["sparql"] for _ in range(RUNS)]

    # All strings must be identical to the first.
    identical = all(s == sparql_runs[0] for s in sparql_runs)
    status = "[PASS]" if identical else "[FAIL]"

    print(f"  {status} {question}")

    if not identical:
        all_deterministic = False
        # Show which runs diverged to help diagnose.
        for i, s in enumerate(sparql_runs):
            if s != sparql_runs[0]:
                print(f"         Run {i+1} produced different SPARQL than run 1.")

print()

print(f"Checking determinism: SPARQL output should be byte-identical across all {RUNS} runs of the same question...")

if not all_deterministic:
    print("VERIFICATION FAILED: ask_graph() produced different SPARQL for the same question across multiple runs.")
    print("Most likely cause: the function is calling a Bedrock generative model (e.g. Claude) for SPARQL")
    print("generation rather than using Titan Embeddings for template selection.")
    print("Check that EMBEDDING_MODEL_ID is set to 'amazon.titan-embed-text-v2:0' and that no code path")
    print("calls InvokeModel on a text-generation model inside ask_graph(). Cell 06 of this notebook")
    print("shows the correct embedding-based pattern.")

assert all_deterministic, (
    "Determinism check failed: ask_graph() produced different SPARQL for the "
    "same question across multiple runs. The most likely cause is that the "
    "function is calling a generative Bedrock model rather than using "
    "Titan Embeddings for template selection. Verify that EMBEDDING_MODEL_ID "
    "is 'amazon.titan-embed-text-v2:0'."
)

print(f"[PASS] All {len(PILOT_QUESTIONS)} questions produced byte-identical SPARQL across {RUNS} runs.")
print()
print("This is the property that makes ATLAS auditable. Every invocation of")
print("ask_graph() with the same question is traceable to the same template,")
print("the same SPARQL, and the same graph state — the three elements an")
print("examiner applying SR 11-7 or OCC 2011-12 would ask to see.")

## What just changed

You have seen the pattern that every Phase 1 agent implements. A natural-language
question enters; an embedding lookup selects the closest validated SPARQL template;
the template executes against the SHACL-governed SLGD; the result comes back with
a `template_id` that serves as the audit anchor. The LLM touched the surface — the
question's meaning — and nothing else. The reasoning happened in the graph.

This is why ATLAS is auditable where most agentic demos are not. The SPARQL query,
the graph it ran against, and the SHACL shapes that validated that graph are all
inspectable artifacts that exist independently of any model. A regulator applying
SR 11-7 or OCC 2011-12 can audit the decision without access to the model's
weights, its training data, or its internal reasoning.

The rest of Phase 1 is making this pattern multi-agent, discoverable, governable,
and surfaced through a UI: wrapping it in MCP (Model Context Protocol) servers so
agents can call capabilities without knowing the implementation, registering those
capabilities in an Agent Registry so a UI can discover them by persona, federating
them through a FIBO (Financial Industry Business Ontology)-shaped GraphQL API, and
finally rendering them in the Wholesale UI that a Consumer Banker actually uses.